In [1]:
import pandas as pd

df = pd.read_csv('Week_3/week_3_final_dataset.csv')

Removing the Live and Time columns as we no longer want to use it

In [2]:
df = df.drop(columns=['Live', 'Time'])

Slightly altered the week 3 eda notebook to re-catagorize subgenres that had fewer than 15 instances, now only 42 total subgenres exist

In [3]:
subgenre_counts = df['Subgenre'].value_counts()
subgenre_counts

Subgenre
hip hop              272
indie rock           207
indie pop            203
pop 80s              168
alternative rock     163
rock 70s             148
rock 80s             144
country              130
pop                  124
rock                 119
electronic            88
rnb                   86
pop rock              75
folk                  67
country pop           63
indie                 62
alternative           59
rock 60s              54
alternative rnb       51
dance                 49
indie folk            48
dance pop             46
pop 70s               46
pop dance             46
folk rock             45
soul                  45
rock alternative      43
hip hop rnb           36
soul 70s              35
rock 90s              31
electronic pop        28
pop 90s               27
electronic dance      27
soul rnb              23
pop 60s               22
indie alternative     19
pop synthpop          19
hip hop 90s           19
pop rnb               19
soul 80s        

Normalizing the numeric features on a 0 to 1 scale

In [4]:
from sklearn.preprocessing import MinMaxScaler

# Example: Select only numeric columns
numeric_cols = df.select_dtypes(include='number').columns

# Create a scaler and fit-transform the numeric columns
scaler = MinMaxScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

Preparing the data for tokenization by cleaning the lyrics, also removing stop words and lemanizing the lyrics

In [6]:
import re
import spacy
from nltk.corpus import stopwords

# Load spaCy model and stopwords
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
stop_words = set(stopwords.words("english"))

def clean_lyrics(text):
    if pd.isnull(text):
        return ""

    # Lowercase
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # Remove [annotations like chorus/verse]
    text = re.sub(r"\[.*?\]", "", text)

    # Remove punctuation (except apostrophes)
    text = re.sub(r"[^a-z0-9'\s]", "", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenize with spaCy
    doc = nlp(text)

    # Lemmatize and remove stopwords
    tokens = [token.lemma_ for token in doc if token.text not in stop_words and token.lemma_ not in stop_words]

    return " ".join(tokens)


/opt/anaconda3/envs/ml/lib/python3.12/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


In [8]:
df['Clean_Lyrics'] = df['Lyrics'].apply(clean_lyrics)

In [10]:
df = df.drop(columns='Lyrics')

In [11]:
df

,Song,Artist,Popularity,BPM,Dance,Energy,Acoustic,Happy,Loud,Camelot,Genre,Subgenre,Clean_Lyrics
0,I'm So Excited,The Pointer Sisters,0.5625,0.252941,0.682353,0.854167,0.101010,0.687500,0.80,4B,pop,pop 80s,tonight 's night go make happen tonight put th...
1,Cheri Cheri Lady,Modern Talking,0.7750,0.382353,0.670588,0.604167,0.464646,0.854167,0.48,6B,pop,pop 80s,oh explain every time oh feel real take heart ...
2,Give It Up,KC & The Sunshine Band,0.4750,0.452941,0.858824,0.635417,0.080808,0.843750,0.56,5B,pop,pop,everybody want everybody want love would like ...
3,It's Raining Men - Single Version,The Weather Girls,0.3750,0.511765,0.647059,0.927083,0.464646,0.447917,0.80,4A,pop,pop,hi hi weather girl uh huh get news well listen...
4,Take on Me,a-ha,0.8625,0.205882,0.541176,0.895833,0.020202,0.885417,0.72,11A,pop,pop 80s,talk away know say say anyway today another da...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3016,Play That Funky Music,Wild Cherry,0.7000,0.352941,0.823529,0.656250,0.040404,0.937500,0.56,11B,soul,soul disco,ahey huh yeah hey heyah boogie singer playing ...
3017,Rock with You - Single Version,Michael Jackson,0.7750,0.382353,0.823529,0.520833,0.181818,0.854167,0.52,3B,pop,pop 80s,girl close eye let rhythm get try fight nothin...
3018,You Sexy Thing,Hot Chocolate,0.6875,0.335294,0.800000,0.718750,0.525253,0.968750,0.84,7B,soul,soul disco,believe miracle sexy thing sexy thing believe ...
3019,Get It On,T. Rex,0.6500,0.458824,0.729412,0.875000,0.181818,0.916667,0.76,10A,rock,rock 70s,well dirty sweet clad black look back love dir...


In [12]:
df.to_csv('Week_4/week_4_final_dataset.csv', index=False)